# Huấn luyện mô hình gốc DeepVQE với bộ VCTK-DEMAND (VoiceBank-DEMAND)
Notebook này thiết lập môi trường để tải bộ dữ liệu, liên kết Google Drive, clone mã nguồn DeepVQE, xử lý metadata (file csv) và huấn luyện mô hình.

## 1. Cài đặt môi trường & Kết nối Google Drive

In [ ]:
!pip install torchaudio speechbrain tensorboard soundfile pandas tqdm einops wandb

from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/DeepVQE_Workspace'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Thư mục làm việc hiện tại: {os.getcwd()}")

## 2. Clone mã nguồn DeepVQE

In [ ]:
# Repo này có sẵn deepvqe.py, ablation/ và stream/ benchmark scripts.
GIT_REPO = "https://github.com/hoxuanphu/Pdeepvqe.git"
REPO_DIR = "deepvqe"

if not os.path.exists(REPO_DIR):
    !git clone {GIT_REPO} {REPO_DIR}
else:
    print(f"Thư mục {REPO_DIR} đã tồn tại.")
    if not os.path.exists(os.path.join(REPO_DIR, "ablation")):
        print("Lưu ý: repo hiện có chưa có ablation/; cell benchmark sẽ tự lấy từ Pdeepvqe nếu cần.")

os.chdir(REPO_DIR)
print(f"Đã vào thư mục mã nguồn: {os.getcwd()}")

## 3. Tải bộ dữ liệu VoiceBank-DEMAND
Dữ liệu gốc được tải từ Đại học Edinburgh (chiếm khoảng 4GB - 5GB sau khi giải nén).

**Lưu ý**: VCTK-DEMAND gốc có sample rate 48kHz. Code training sẽ tự resample xuống 16kHz.

In [ ]:
import os
import zipfile

data_dir = "/content/drive/MyDrive/DeepVQE_Workspace/data/voicebank-demand"
os.makedirs(data_dir, exist_ok=True)

datasets = {
    "clean_trainset_28spk_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip",
    "noisy_trainset_28spk_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip",
    "clean_testset_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip",
    "noisy_testset_wav": "https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip"
}

for folder_name, url in datasets.items():
    extract_path = os.path.join(data_dir, folder_name)
    zip_path = os.path.join(data_dir, folder_name + ".zip")
    
    # Kiểm tra xem thư mục giải nén đã có dữ liệu chưa
    if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
        print(f"Thư mục {folder_name} đã tồn tại và có dữ liệu trong Drive, bỏ qua tải.")
    else:
        if not os.path.exists(zip_path):
            print(f"Đang tải {folder_name}.zip về Drive...")
            !wget -nc -P {data_dir} {url}
        else:
            print(f"File {folder_name}.zip đã tồn tại trong Drive, tiến hành giải nén.")
        
        print(f"Đang giải nén {folder_name}.zip...")
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            # (Tùy chọn) Xóa file zip để tiết kiệm dung lượng Drive
            # os.remove(zip_path)
            print(f"Đã giải nén xong {folder_name}.zip")
        except zipfile.BadZipFile:
            print(f"Lỗi: File {folder_name}.zip bị hỏng. Vui lòng xóa file này và chạy lại ô này.")

print("\nKiểm tra và chuẩn bị dữ liệu VCTK-DEMAND hoàn tất!")

## 4. Xử lý và phân chia Dataset (Tạo file CSV)
Để framework (ví dụ như SpeechBrain) dễ dàng tải dữ liệu, ta sẽ tạo các file metadata dạng CSV (`train.csv`, `valid.csv`, `test.csv`).

In [ ]:
import glob
import pandas as pd

def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(os.path.join(clean_dir, "*.wav")))
    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    
    assert len(clean_files) == len(noisy_files), f"Số lượng file không khớp! ({len(clean_files)} != {len(noisy_files)})"
    
    data = []
    for c, n in zip(clean_files, noisy_files):
        filename = os.path.basename(c)
        data.append({
            "ID": filename.replace(".wav", ""),
            "clean_wav": os.path.abspath(c),
            "noisy_wav": os.path.abspath(n)
        })
        
    df = pd.DataFrame(data)
    df.to_csv(output_csv, index=False)
    print(f"Đã tạo {output_csv} với {len(df)} mẫu.")

base_dir = data_dir
# Tạo file CSV thô
create_csv(f"{base_dir}/clean_trainset_28spk_wav", f"{base_dir}/noisy_trainset_28spk_wav", f"{base_dir}/train_full.csv")
create_csv(f"{base_dir}/clean_testset_wav", f"{base_dir}/noisy_testset_wav", f"{base_dir}/test.csv")

# Tách train_full thành train (90%) và valid (10%)
df_train_full = pd.read_csv(f"{base_dir}/train_full.csv")
df_train_full = df_train_full.sample(frac=1, random_state=42).reset_index(drop=True)

split_idx = int(len(df_train_full) * 0.90)
df_train_full.iloc[:split_idx].to_csv(f"{base_dir}/train.csv", index=False)
df_train_full.iloc[split_idx:].to_csv(f"{base_dir}/valid.csv", index=False)

print("Hoàn tất tạo metadata (train.csv, valid.csv, test.csv)!")

## 5. Cấu hình Train (hparams.yaml)
File cấu hình hyperparameters cho DeepVQE.

**Lưu ý**: Đã bỏ `optimizer` khỏi `recoverables` vì `opt_class` là class chưa khởi tạo — SpeechBrain Brain sẽ tự tạo và quản lý optimizer instance.

In [ ]:
yaml_content = """
seed: 1234
__set_seed: !apply:torch.manual_seed [!ref <seed>]

# Data paths
data_folder: /content/drive/MyDrive/DeepVQE_Workspace/data/voicebank-demand
train_annotation: !ref <data_folder>/train.csv
valid_annotation: !ref <data_folder>/valid.csv
test_annotation: !ref <data_folder>/test.csv

# Audio parameters
sample_rate: 16000
n_fft: 512
hop_length: 256
win_length: 512

# Training parameters
batch_size: 4
N_epochs: 100
lr: 0.0002
grad_clip: 5.0

# Loss weights
loss_waveform_weight: 0.5
loss_spectral_weight: 0.5

# Dataloaders
dataloader_options:
    batch_size: !ref <batch_size>
    num_workers: 2
    drop_last: False

# Model Definition
model: !new:deepvqe.DeepVQE

# Optimizer
opt_class: !name:torch.optim.Adam
    lr: !ref <lr>

epoch_counter: !new:speechbrain.utils.epoch_loop.EpochCounter
    limit: !ref <N_epochs>

# Checkpointer - lưu lại mô hình trực tiếp lên Google Drive
# Lưu ý: KHÔNG lưu optimizer ở đây vì opt_class là class, không phải instance.
# SpeechBrain Brain.fit() tự quản lý optimizer.
output_folder: /content/drive/MyDrive/DeepVQE_Workspace/checkpoints/deepvqe_vctk/<seed>
save_folder: !ref <output_folder>/save
checkpointer: !new:speechbrain.utils.checkpoints.Checkpointer
    checkpoints_dir: !ref <save_folder>
    recoverables:
        model: !ref <model>
        counter: !ref <epoch_counter>
"""

with open("hparams_vctk.yaml", "w") as f:
    f.write(yaml_content)
print("Đã lưu cấu hình vào hparams_vctk.yaml")

## 6. Code Train
Training loop sử dụng SpeechBrain `Brain` class.

**Các sửa đổi so với phiên bản cũ:**
- ✅ Thêm resample 16kHz trong `audio_pipeline` (VCTK-DEMAND gốc là 48kHz)
- ✅ Tạo `hann_window` một lần trong `on_stage_start` thay vì mỗi forward call
- ✅ Sửa length mismatch giữa predictions và clean_wavs
- ✅ Thêm spectral loss (STFT magnitude L1) bên cạnh waveform L1
- ✅ Thêm gradient clipping cho training ổn định
- ✅ Sửa lỗi checkpointer (bỏ optimizer khỏi recoverables)

In [ ]:
import os
import sys
import torch
import torchaudio
from speechbrain.core import Brain
from speechbrain.dataio.dataset import DynamicItemDataset
from hyperpyyaml import load_hyperpyyaml

# Đảm bảo đang đứng đúng thư mục chứa mã nguồn
work_dir = '/content/drive/MyDrive/DeepVQE_Workspace/deepvqe'
if os.path.exists(work_dir):
    os.chdir(work_dir)
    if work_dir not in sys.path:
        sys.path.insert(0, work_dir)

# Đọc cấu hình từ file YAML
with open('hparams_vctk.yaml') as fin:
    hparams = load_hyperpyyaml(fin)

# ============================================================
# 1. Chuẩn bị DataLoader (SpeechBrain DynamicItemDataset)
# ============================================================

TARGET_SR = hparams.get('sample_rate', 16000)

def audio_pipeline(file_path):
    """Đọc file audio và resample về TARGET_SR nếu cần."""
    sig, sr = torchaudio.load(file_path)
    if sr != TARGET_SR:
        sig = torchaudio.functional.resample(sig, sr, TARGET_SR)
    return sig.squeeze(0)  # [1, T] -> [T] để SpeechBrain tự padding

train_set = DynamicItemDataset.from_csv(hparams['train_annotation'])
valid_set = DynamicItemDataset.from_csv(hparams['valid_annotation'])

for dataset in [train_set, valid_set]:
    dataset.add_dynamic_item(audio_pipeline, takes="noisy_wav", provides="noisy_sig")
    dataset.add_dynamic_item(audio_pipeline, takes="clean_wav", provides="clean_sig")
    dataset.set_output_keys(["id", "noisy_sig", "clean_sig"])

# ============================================================
# 2. Định nghĩa Recipe Train (kế thừa từ speechbrain.Brain)
# ============================================================

class VQE_Brain(Brain):
    
    def on_stage_start(self, stage, epoch=None):
        """Tạo hann_window MỘT LẦN cho mỗi stage, tránh tạo lại trong mỗi forward."""
        super().on_stage_start(stage, epoch)
        
        # Sử dụng getattr với giá trị mặc định để tránh lỗi nếu quên chạy lại cell 5
        win_length = getattr(self.hparams, 'win_length', 512)
        self.window = torch.hann_window(win_length).to(self.device)
        self.n_fft = getattr(self.hparams, 'n_fft', 512)
        self.hop_length = getattr(self.hparams, 'hop_length', 256)
        
        # Khởi tạo tracking loss cho validation
        if stage != "train":
            self.valid_loss_sum = 0.0
            self.valid_loss_count = 0

    def compute_forward(self, batch, stage):
        noisy_wavs, lens = batch.noisy_sig
        noisy_wavs = noisy_wavs.to(self.device)
        
        # STFT: (B, T) -> (B, F, T_frames, 2)
        noisy_stft = torch.stft(
            noisy_wavs, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            win_length=self.n_fft,
            window=self.window, 
            return_complex=True
        )
        noisy_stft_real = torch.view_as_real(noisy_stft)  # (B, F, T_frames, 2)
        
        # DeepVQE: (B, F, T_frames, 2) -> (B, F, T_frames, 2)
        pred_stft_real = self.modules.model(noisy_stft_real)
        
        # ISTFT: khôi phục waveform, cắt về đúng length ban đầu
        pred_stft_complex = torch.complex(pred_stft_real[..., 0], pred_stft_real[..., 1])
        enhanced_wavs = torch.istft(
            pred_stft_complex, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            win_length=self.n_fft,
            window=self.window, 
            length=noisy_wavs.shape[1]
        )
        
        # Trả về cả waveform và STFT để tính multi-domain loss
        return enhanced_wavs, pred_stft_real

    def compute_objectives(self, predictions, batch, stage):
        enhanced_wavs, pred_stft = predictions
        clean_wavs, lens = batch.clean_sig
        clean_wavs = clean_wavs.to(self.device)
        
        # === Fix length mismatch ===
        # Sau STFT->model->ISTFT, enhanced_wavs có thể khác length với clean_wavs
        min_len = min(enhanced_wavs.shape[-1], clean_wavs.shape[-1])
        enhanced_wavs = enhanced_wavs[..., :min_len]
        clean_wavs = clean_wavs[..., :min_len]
        
        # --- Loss 1: Waveform L1 ---
        loss_wav = torch.nn.functional.l1_loss(enhanced_wavs, clean_wavs)
        
        # --- Loss 2: Spectral Magnitude L1 (STFT domain) ---
        # Tính STFT của clean signal để so sánh trên spectral domain
        clean_stft = torch.stft(
            clean_wavs, 
            n_fft=self.n_fft, 
            hop_length=self.hop_length, 
            win_length=self.n_fft,
            window=self.window, 
            return_complex=True
        )
        clean_stft_real = torch.view_as_real(clean_stft)  # (B, F, T_frames, 2)
        
        # Đảm bảo T_frames khớp giữa pred và clean
        min_t = min(pred_stft.shape[2], clean_stft_real.shape[2])
        pred_stft_t = pred_stft[:, :, :min_t, :]
        clean_stft_t = clean_stft_real[:, :, :min_t, :]
        
        # L1 trên magnitude spectrum
        pred_mag = torch.sqrt(pred_stft_t[..., 0]**2 + pred_stft_t[..., 1]**2 + 1e-12)
        clean_mag = torch.sqrt(clean_stft_t[..., 0]**2 + clean_stft_t[..., 1]**2 + 1e-12)
        loss_spec = torch.nn.functional.l1_loss(pred_mag, clean_mag)
        
        # --- Combined Loss ---
        # Sử dụng getattr để lấy giá trị mặc định, tránh lỗi cũ
        w_wav = getattr(self.hparams, 'loss_waveform_weight', 0.5)
        w_spec = getattr(self.hparams, 'loss_spectral_weight', 0.5)
        loss = w_wav * loss_wav + w_spec * loss_spec
        
        # Track validation loss
        if stage != "train":
            self.valid_loss_sum += loss.item()
            self.valid_loss_count += 1
        
        return loss
    
    def on_stage_end(self, stage, stage_loss, epoch=None):
        """Log kết quả."""
        super().on_stage_end(stage, stage_loss, epoch)
        
        import wandb
        if wandb.run is not None:
            wandb.log({f"{stage}_loss": stage_loss, "epoch": epoch})
            
        if stage == "valid":
            avg_valid_loss = self.valid_loss_sum / max(self.valid_loss_count, 1)
            print(f"  Epoch {epoch} | Valid Loss: {avg_valid_loss:.6f}")

# ============================================================
# 3. Khởi tạo và Train
# ============================================================

vqe_brain = VQE_Brain(
    modules={'model': hparams['model']},
    opt_class=hparams['opt_class'],
    hparams=hparams,
    checkpointer=hparams['checkpointer'], 
)

# Chạy Train Loop (tự động Resume từ checkpoint nếu có)
# Khởi tạo wandb
import wandb
# Nếu chạy trên Colab, hàm wandb.login() sẽ yêu cầu API key nếu chưa đăng nhập.
wandb.login()
wandb.init(
    project="DeepVQE", 
    name="train_base_deepvqe",
    config={
        "learning_rate": hparams.get("lr", 0.0002),
        "epochs": hparams.get("N_epochs", 100),
        "batch_size": hparams.get("batch_size", 4),
        "loss_waveform_weight": hparams.get("loss_waveform_weight", 0.5),
        "loss_spectral_weight": hparams.get("loss_spectral_weight", 0.5)
    }
)

vqe_brain.fit(
    vqe_brain.hparams.epoch_counter,
    train_set,
    valid_set,
    train_loader_kwargs=hparams.get('dataloader_options', {'batch_size': 4}),
    valid_loader_kwargs=hparams.get('dataloader_options', {'batch_size': 4}),
)
print("\n=== Huấn luyện hoàn tất! ===")
print(f"Checkpoint đã được lưu tại: {hparams.get('save_folder', 'save')}")
# Kết thúc wandb
wandb.finish()


## 7. Kiểm tra nhanh sau training
Chạy inference trên một mẫu test để xác nhận model hoạt động.

In [ ]:
import torchaudio
import IPython.display as ipd

# --- Khôi phục hparams nếu kernel bị restart ---
if 'hparams' not in globals():
    print('Biến hparams chưa được định nghĩa. Đang tải lại từ hparams_vctk.yaml...')
    import sys, os
    from hyperpyyaml import load_hyperpyyaml
    work_dir = globals().get('WORK_DIR', '/content/drive/MyDrive/DeepVQE_Workspace')
    repo_dir = os.path.join(work_dir, 'deepvqe')
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
        if repo_dir not in sys.path:
            sys.path.insert(0, repo_dir)
    else:
        print('Cảnh báo: Không tìm thấy thư mục mã nguồn. Vui lòng chạy lại các Cell trên cùng để mount Drive và clone repo.')
    try:
        import deepvqe
        with open('hparams_vctk.yaml') as fin:
            hparams = load_hyperpyyaml(fin)
        if 'checkpointer' in hparams:
            print('Khôi phục trọng số mô hình từ checkpoint...')
            hparams['checkpointer'].recover_if_possible()
    except FileNotFoundError:
        print('Lỗi: Không tìm thấy file hparams_vctk.yaml! Vui lòng chạy lại Cell 5.')
        raise
    except ImportError as e:
        print('Lỗi import thư viện:', e)
        print('Vui lòng đảm bảo bạn đã chạy Cell 1 để cài đặt các thư viện (einops, speechbrain...)')
        raise
TARGET_SR = hparams.get('sample_rate', 16000)

# Load một file test mẫu
test_csv = hparams.get('test_annotation', os.path.join(globals().get('data_dir', '/content/drive/MyDrive/DeepVQE_Workspace/data/voicebank-demand'), 'test.csv'))
import pandas as pd
df_test = pd.read_csv(test_csv)
sample = df_test.iloc[0]

noisy_path = sample['noisy_wav']
clean_path = sample['clean_wav']

sig_noisy, sr = torchaudio.load(noisy_path)
if sr != TARGET_SR:
    sig_noisy = torchaudio.functional.resample(sig_noisy, sr, TARGET_SR)

# Inference
model = hparams['model'].eval().to('cuda' if torch.cuda.is_available() else 'cpu')
device = next(model.parameters()).device
window = torch.hann_window(hparams.get('n_fft', 512)).to(device)

with torch.no_grad():
    noisy_wav = sig_noisy.squeeze(0).to(device)
    stft = torch.stft(noisy_wav, n_fft=512, hop_length=256, win_length=512, 
                      window=window, return_complex=True)
    stft_real = torch.view_as_real(stft).unsqueeze(0)  # (1, F, T, 2)
    pred_stft = model(stft_real)
    pred_complex = torch.complex(pred_stft[..., 0], pred_stft[..., 1]).squeeze(0)
    enhanced = torch.istft(pred_complex, n_fft=512, hop_length=256, win_length=512,
                           window=window, length=noisy_wav.shape[0])

print(f"Sample: {sample['ID']}")
print(f"Noisy:")
ipd.display(ipd.Audio(sig_noisy.squeeze().cpu().numpy(), rate=TARGET_SR))
print(f"Enhanced:")
ipd.display(ipd.Audio(enhanced.cpu().numpy(), rate=TARGET_SR))

sig_clean, sr_c = torchaudio.load(clean_path)
if sr_c != TARGET_SR:
    sig_clean = torchaudio.functional.resample(sig_clean, sr_c, TARGET_SR)
print(f"Clean (ground truth):")
ipd.display(ipd.Audio(sig_clean.squeeze().cpu().numpy(), rate=TARGET_SR))

## 8. Chạy ablation benchmarks trên Colab
Cell này chạy smoke checks, benchmark kiến trúc cho các cấu hình trong `ablation/`, và có thể bật thêm train/eval/ONNX khi cần.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== TÙY CHỈNH NHANH =====================
WORKSPACE = Path('/content/drive/MyDrive/DeepVQE_Workspace')
AB_SOURCE_DIR = None  # ví dụ: '/content/drive/MyDrive/deepvqe/ablation'; None = tự tìm/clone Pdeepvqe
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/content/Pdeepvqe_benchmark_src')

# Nếu muốn chạy ít cấu hình để smoke test trước, sửa thành: ['Baseline', 'B1a']
AB_CONFIGS = [
    'Baseline', 'B1a', 'B1b', 'B2', 'B3',
    'C1', 'C1a-g2', 'C1a-g4', 'C1b', 'C2', 'C3', 'C4',
]

RUN_SMOKE_TESTS = True
RUN_ARCH_BENCHMARK = True

# None = tự tìm best.pt trong Drive/Colab. Nếu có nhiều best.pt, điền path cụ thể vào đây.
BASELINE_CHECKPOINT = None

# Hai phần dưới tốn GPU lâu hơn. Bật True khi muốn benchmark chất lượng sau train.
RUN_TRAINING = False
RUN_QUALITY_EVAL = True
RUN_ONNX_EXPORT = False

AB_EPOCHS = 1
AB_BATCH_SIZE = 8
AB_NUM_WORKERS = 0
AB_PIN_MEMORY = False
AB_EARLY_STOP_PATIENCE = 12
AB_EARLY_STOP_MIN_DELTA = 0.01
AB_EARLY_STOP_MIN_EPOCHS = 20
AB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AB_USE_DATA_PARALLEL = torch.cuda.is_available() and torch.cuda.device_count() > 1
RESUME_EXISTING = True

ARCH_FRAMES = 63
ARCH_WARMUP = 1
ARCH_REPEATS = 3
INSTALL_OPTIONAL_PACKAGES = False  # ptflops / pesq / pystoi / onnxruntime nếu cần
# ===========================================================

repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = WORKSPACE / 'deepvqe'
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

print(f'Repo dir: {repo_dir}')
print(f'Device: {AB_DEVICE}')
if torch.cuda.is_available():
    print(f'CUDA GPUs: {torch.cuda.device_count()}')
    for gpu_idx in range(torch.cuda.device_count()):
        print(f'  GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}')
print(f'DataParallel: {AB_USE_DATA_PARALLEL}')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source(explicit_dir=None):
    if explicit_dir:
        path = Path(explicit_dir)
        if not (path / 'deepvqe_ablation.py').exists():
            raise FileNotFoundError(f'AB_SOURCE_DIR không hợp lệ: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    for root in roots:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if BENCHMARK_REPO_DIR.exists() and (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'deepvqe_ablation.py').exists():
    source_ablation = find_ablation_source(AB_SOURCE_DIR)
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/deepvqe_ablation.py. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, ablation streaming benchmark sẽ không chạy được.')

def patch_eval_script_for_uploaded_checkpoints():
    path = repo_dir / 'ablation' / 'eval_ablation_quality.py'
    if not path.exists():
        return
    text = path.read_text(encoding='utf-8')
    original = text
    text = text.replace(
        'ckpt = torch.load(str(checkpoint_path), map_location=device)',
        "ckpt = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)",
    )
    text = text.replace(
        'model.load_state_dict(ckpt["model"])',
        "state = ckpt.get('model', ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)))\n"
        "    state = {k.replace('module.', '', 1): v for k, v in state.items()}\n"
        "    model.load_state_dict(state, strict=True)",
    )
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Đã patch {path} để load checkpoint upload an toàn hơn.')

patch_eval_script_for_uploaded_checkpoints()

results_dir = WORKSPACE / 'results/ablation'
runs_dir = WORKSPACE / 'runs/ablation'
onnx_dir = WORKSPACE / 'onnx_models/ablation'
manifest_dir = WORKSPACE / 'metadata/ablation_manifests'
for path in [results_dir, runs_dir, onnx_dir, manifest_dir]:
    path.mkdir(parents=True, exist_ok=True)

arch_csv = results_dir / 'ablation_arch_benchmark.csv'
quality_csv = results_dir / 'ablation_quality.csv'
onnx_csv = results_dir / 'ablation_onnx.csv'
summary_csv = results_dir / 'ablation_summary.csv'

def find_best_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy BASELINE_CHECKPOINT: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob('best.pt'))
        if candidates:
            break
    if not candidates:
        return None

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

uploaded_baseline_ckpt = find_best_checkpoint(BASELINE_CHECKPOINT)
if uploaded_baseline_ckpt is not None:
    print(f'Dùng Baseline checkpoint: {uploaded_baseline_ckpt}')
else:
    print('Không tìm thấy best.pt trong Drive/Colab; Baseline eval/ONNX sẽ dùng checkpoint trong runs_dir nếu có.')

def checkpoint_for_config(config_id):
    if config_id == 'Baseline' and uploaded_baseline_ckpt is not None:
        return uploaded_baseline_ckpt
    return runs_dir / config_id / 'best.pt'

if INSTALL_OPTIONAL_PACKAGES:
    packages = ['ptflops']
    if RUN_QUALITY_EVAL:
        packages += ['pesq', 'pystoi']
    if RUN_ONNX_EXPORT:
        packages += ['onnx', 'onnxruntime', 'onnxsim']
    run_py(['-m', 'pip', 'install', '-q', *packages])

def csv_to_jsonl(csv_path, jsonl_path):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path

metadata_base = Path(globals().get('data_dir', str(WORKSPACE / 'data/voicebank-demand')))
csv_manifests = {
    'train': metadata_base / 'train.csv',
    'valid': metadata_base / 'valid.csv',
    'test': metadata_base / 'test.csv',
}
jsonl_manifests = {}
if all(path.exists() for path in csv_manifests.values()):
    for split, csv_path in csv_manifests.items():
        jsonl_manifests[split] = csv_to_jsonl(csv_path, manifest_dir / f'{split}.jsonl')
    print('Đã tạo manifest JSONL cho ablation:')
    for split, path in jsonl_manifests.items():
        print(f'  {split}: {path}')
elif RUN_TRAINING or RUN_QUALITY_EVAL:
    raise FileNotFoundError('Thiếu train/valid/test CSV. Hãy chạy cell tạo metadata trước.')
else:
    print('Chưa có metadata CSV; bỏ qua manifest train/eval và chỉ chạy benchmark kiến trúc.')

if RUN_SMOKE_TESTS:
    run_py(['ablation/test_ablation_baseline.py'])
    run_py(['ablation/test_ablation_streaming.py', '--configs', *AB_CONFIGS])

if RUN_ARCH_BENCHMARK:
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', arch_csv,
        '--configs', *AB_CONFIGS,
        '--device', AB_DEVICE,
        '--frames', ARCH_FRAMES,
        '--warmup', ARCH_WARMUP,
        '--repeats', ARCH_REPEATS,
    ])

if RUN_TRAINING:
    for config_id in AB_CONFIGS:
        output_dir = runs_dir / config_id
        args = [
            'ablation/train_ablation.py',
            '--config-id', config_id,
            '--train-manifest', jsonl_manifests['train'],
            '--valid-manifest', jsonl_manifests['valid'],
            '--output-dir', output_dir,
            '--device', AB_DEVICE,
            '--epochs', AB_EPOCHS,
            '--batch-size', AB_BATCH_SIZE,
            '--num-workers', AB_NUM_WORKERS,
            '--early-stop-patience', AB_EARLY_STOP_PATIENCE,
            '--early-stop-min-delta', AB_EARLY_STOP_MIN_DELTA,
            '--early-stop-min-epochs', AB_EARLY_STOP_MIN_EPOCHS,
        ]
        last_ckpt = output_dir / 'last.pt'
        if not AB_PIN_MEMORY:
            args += ['--no-pin-memory']
        if AB_USE_DATA_PARALLEL:
            args += ['--data-parallel']
        if RESUME_EXISTING and last_ckpt.exists():
            args += ['--resume', last_ckpt, '--ignore-bad-resume']
        run_py(args)

if RUN_QUALITY_EVAL:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua eval {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/eval_ablation_quality.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--manifest', jsonl_manifests['test'],
            '--output', quality_csv,
            '--device', AB_DEVICE,
        ])

if RUN_ONNX_EXPORT:
    for config_id in AB_CONFIGS:
        ckpt = checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua ONNX {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/export_ablation_onnx.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--output-dir', onnx_dir,
            '--results', onnx_csv,
            '--device', 'cpu',
        ])

run_py([
    'ablation/collect_ablation_results.py',
    '--arch', arch_csv,
    '--quality', quality_csv,
    '--onnx', onnx_csv,
    '--output', summary_csv,
])

print('\nCSV kết quả:')
for csv_path in [arch_csv, quality_csv, onnx_csv, summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path).head(30))

archive_base = Path('/content/ablation_benchmark_results')
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(results_dir))
print(f'Đã nén kết quả benchmark: {archive_path}')
display(FileLink(str(archive_path)))

## 9. Quality benchmark với `eval_ablation_quality.py` trên Colab
Cell này chỉ chạy benchmark chất lượng cho checkpoint `best.pt`, dùng cùng script `ablation/eval_ablation_quality.py`.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== TÙY CHỈNH NHANH =====================
WORKSPACE = Path('/content/drive/MyDrive/DeepVQE_Workspace')
EVAL_CONFIG_ID = 'Baseline'
EVAL_CHECKPOINT = None  # None = tự tìm best.pt trong Drive/Colab
EVAL_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_MAX_TEST_ITEMS = None  # ví dụ 50 để smoke test nhanh; None = full test set
INSTALL_PESQ_STOI = False  # True nếu muốn thêm PESQ/STOI ngoài SI-SDR
AB_SOURCE_DIR = None  # ví dụ: '/content/drive/MyDrive/deepvqe/ablation'; None = tự tìm/clone Pdeepvqe
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/content/Pdeepvqe_benchmark_src')

EVAL_OUTPUT_CSV = WORKSPACE / 'results/ablation/ablation_quality_uploaded_model.csv'
EVAL_MANIFEST_JSONL = WORKSPACE / 'metadata/ablation_manifests/test_eval_ablation_quality.jsonl'
# ===========================================================

repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = WORKSPACE / 'deepvqe'
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source(explicit_dir=None):
    if explicit_dir:
        path = Path(explicit_dir)
        if not (path / 'deepvqe_ablation.py').exists():
            raise FileNotFoundError(f'AB_SOURCE_DIR không hợp lệ: {path}')
        return path
    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    for root in roots:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if BENCHMARK_REPO_DIR.exists() and (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'eval_ablation_quality.py').exists():
    source_ablation = find_ablation_source(AB_SOURCE_DIR)
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/eval_ablation_quality.py. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, cần cho DeepVQE_Ablation import.')

def patch_eval_script_for_uploaded_checkpoints():
    path = repo_dir / 'ablation' / 'eval_ablation_quality.py'
    if not path.exists():
        return
    text = path.read_text(encoding='utf-8')
    original = text
    text = text.replace(
        'ckpt = torch.load(str(checkpoint_path), map_location=device)',
        "ckpt = torch.load(str(checkpoint_path), map_location='cpu', weights_only=False)",
    )
    text = text.replace(
        'model.load_state_dict(ckpt["model"])',
        "state = ckpt.get('model', ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt)))\n"
        "    state = {k.replace('module.', '', 1): v for k, v in state.items()}\n"
        "    model.load_state_dict(state, strict=True)",
    )
    if text != original:
        path.write_text(text, encoding='utf-8')
        print(f'Đã patch {path} để load checkpoint upload an toàn hơn.')

patch_eval_script_for_uploaded_checkpoints()

def find_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy EVAL_CHECKPOINT: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob('best.pt'))
        if candidates:
            break
    if not candidates:
        raise FileNotFoundError('Không tìm thấy best.pt trong Drive/Colab. Hãy set EVAL_CHECKPOINT.')

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

def test_csv_to_jsonl(csv_path, jsonl_path, max_items=None):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    if max_items is not None:
        df = df.head(int(max_items))

    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path, len(df)

if INSTALL_PESQ_STOI:
    run_py(['-m', 'pip', 'install', '-q', 'pesq', 'pystoi'])

metadata_base = Path(globals().get('data_dir', str(WORKSPACE / 'data/voicebank-demand')))
test_csv = metadata_base / 'test.csv'
if not test_csv.exists():
    raise FileNotFoundError('Không tìm thấy test.csv. Hãy chạy cell tạo metadata trước.')

checkpoint_path = find_checkpoint(EVAL_CHECKPOINT)
manifest_path, manifest_count = test_csv_to_jsonl(test_csv, EVAL_MANIFEST_JSONL, EVAL_MAX_TEST_ITEMS)
EVAL_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

print(f'Repo dir: {repo_dir}')
print(f'Checkpoint: {checkpoint_path}')
print(f'Manifest: {manifest_path} ({manifest_count} mẫu)')
print(f'Output CSV: {EVAL_OUTPUT_CSV}')
print(f'Device: {EVAL_DEVICE}')

run_py([
    'ablation/eval_ablation_quality.py',
    '--config-id', EVAL_CONFIG_ID,
    '--checkpoint', checkpoint_path,
    '--manifest', manifest_path,
    '--output', EVAL_OUTPUT_CSV,
    '--device', EVAL_DEVICE,
])

df_quality = pd.read_csv(EVAL_OUTPUT_CSV)
display(df_quality)
display(FileLink(str(EVAL_OUTPUT_CSV)))

## 10. Phase 1 Training: B1a, B1b, C1 trên Colab

Cell này train vòng 1 theo `ablation/docs/phase1_training_plan.md`, lưu artifact theo run tag không có ngày tháng, và tự dùng DataParallel nếu runtime Colab có nhiều GPU.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import FileLink, display

# ===================== PHASE 1 TRAINING =====================
WORKSPACE = Path('/content/drive/MyDrive/DeepVQE_Workspace')
PHASE1_RUN_TAG = 'phase1_b1b_vctk_demand_e80_bs8'
PHASE1_TRAIN_CONFIGS = ['B1b']
PHASE1_ARCH_CONFIGS = ['Baseline', *PHASE1_TRAIN_CONFIGS]

PHASE1_RUN_SMOKE_TESTS = True
PHASE1_RUN_ARCH_BENCHMARK = True
PHASE1_RUN_TRAINING = True
PHASE1_RUN_QUALITY_EVAL = True
PHASE1_RUN_ONNX_EXPORT = False

PHASE1_EPOCHS = 80
PHASE1_BATCH_SIZE = 8
PHASE1_NUM_WORKERS = 0
PHASE1_PIN_MEMORY = False
PHASE1_EARLY_STOP_PATIENCE = 12
PHASE1_EARLY_STOP_MIN_DELTA = 0.01
PHASE1_EARLY_STOP_MIN_EPOCHS = 20
PHASE1_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PHASE1_USE_DATA_PARALLEL = torch.cuda.is_available() and torch.cuda.device_count() > 1
PHASE1_RESUME_EXISTING = True
PHASE1_INSTALL_OPTIONAL_PACKAGES = True
PHASE1_EVAL_BASELINE_IF_FOUND = True
PHASE1_BASELINE_CHECKPOINT = None
PHASE1_EVAL_MAX_TEST_ITEMS = None

PHASE1_ARCH_FRAMES = 63
PHASE1_ARCH_WARMUP = 1
PHASE1_ARCH_REPEATS = 3
AB_SOURCE_DIR = None
BENCHMARK_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
BENCHMARK_REPO_DIR = Path('/content/Pdeepvqe_benchmark_src')
# ============================================================

repo_dir = Path.cwd()
if not (repo_dir / 'deepvqe.py').exists():
    fallback_repo = WORKSPACE / 'deepvqe'
    if (fallback_repo / 'deepvqe.py').exists():
        repo_dir = fallback_repo
        os.chdir(repo_dir)
    else:
        raise FileNotFoundError('Không tìm thấy deepvqe.py. Hãy chạy cell clone repo trước.')

print(f'Repo dir: {repo_dir}')
print(f'Run tag: {PHASE1_RUN_TAG}')
print(f'Device: {PHASE1_DEVICE}')
if torch.cuda.is_available():
    print(f'CUDA GPUs: {torch.cuda.device_count()}')
    for gpu_idx in range(torch.cuda.device_count()):
        print(f'  GPU {gpu_idx}: {torch.cuda.get_device_name(gpu_idx)}')
print(f'DataParallel: {PHASE1_USE_DATA_PARALLEL}')

def run_py(args, cwd=repo_dir):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd))
    if result.returncode != 0:
        raise RuntimeError('Lệnh lỗi với exit code ' + str(result.returncode) + ': ' + ' '.join(cmd))
    return result

def find_ablation_source(explicit_dir=None):
    if explicit_dir:
        path = Path(explicit_dir)
        if not (path / 'deepvqe_ablation.py').exists():
            raise FileNotFoundError(f'AB_SOURCE_DIR không hợp lệ: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    for root in roots:
        if not root.exists():
            continue
        for marker in root.rglob('deepvqe_ablation.py'):
            if marker.parent.name == 'ablation':
                return marker.parent

    print(f'Không tìm thấy ablation local; clone benchmark repo: {BENCHMARK_REPO}')
    if not (BENCHMARK_REPO_DIR / 'ablation' / 'deepvqe_ablation.py').exists():
        if (BENCHMARK_REPO_DIR / '.git').exists():
            subprocess.run(['git', '-C', str(BENCHMARK_REPO_DIR), 'pull', '--ff-only'], check=False)
        elif not BENCHMARK_REPO_DIR.exists():
            subprocess.run(['git', 'clone', '--depth', '1', BENCHMARK_REPO, str(BENCHMARK_REPO_DIR)], check=True)

    source = BENCHMARK_REPO_DIR / 'ablation'
    if (source / 'deepvqe_ablation.py').exists():
        return source
    return None

if not (repo_dir / 'ablation' / 'deepvqe_ablation.py').exists():
    source_ablation = find_ablation_source(AB_SOURCE_DIR)
    if source_ablation is None:
        raise FileNotFoundError(
            'Không tìm thấy ablation/deepvqe_ablation.py trong repo hiện tại. '
            f'Không clone được benchmark repo: {BENCHMARK_REPO}'
        )
    print(f'Copy ablation từ: {source_ablation}')
    shutil.copytree(source_ablation, repo_dir / 'ablation', dirs_exist_ok=True)
    source_root = source_ablation.parent
    if (source_root / 'stream').exists() and not (repo_dir / 'stream').exists():
        shutil.copytree(source_root / 'stream', repo_dir / 'stream', dirs_exist_ok=True)

if not (repo_dir / 'stream' / 'modules' / 'convolution.py').exists():
    raise FileNotFoundError('Thiếu stream/modules/convolution.py, ablation streaming benchmark sẽ không chạy được.')

if PHASE1_INSTALL_OPTIONAL_PACKAGES:
    packages = []
    if PHASE1_RUN_ARCH_BENCHMARK:
        packages.append('ptflops')
    if PHASE1_RUN_QUALITY_EVAL:
        packages += ['pesq', 'pystoi']
    if PHASE1_RUN_ONNX_EXPORT:
        packages += ['onnx', 'onnxruntime', 'onnxsim']
    packages = list(dict.fromkeys(packages))
    if packages:
        try:
            run_py(['-m', 'pip', 'install', '-q', *packages])
        except RuntimeError as exc:
            print(f'Cảnh báo: cài optional packages lỗi: {exc}')
            print('Vẫn tiếp tục; metric phụ thuộc package thiếu có thể bị bỏ trống.')

phase1_results_dir = WORKSPACE / 'results' / 'ablation' / PHASE1_RUN_TAG
phase1_runs_dir = WORKSPACE / 'runs' / 'ablation' / PHASE1_RUN_TAG
phase1_onnx_dir = WORKSPACE / 'onnx_models' / 'ablation' / PHASE1_RUN_TAG
phase1_manifest_dir = WORKSPACE / 'metadata' / 'ablation_manifests' / PHASE1_RUN_TAG
for path in [phase1_results_dir, phase1_runs_dir, phase1_onnx_dir, phase1_manifest_dir]:
    path.mkdir(parents=True, exist_ok=True)

phase1_arch_csv = phase1_results_dir / 'ablation_arch_benchmark.csv'
phase1_quality_csv = phase1_results_dir / 'ablation_quality.csv'
phase1_onnx_csv = phase1_results_dir / 'ablation_onnx.csv'
phase1_summary_csv = phase1_results_dir / 'ablation_summary.csv'

def csv_to_jsonl(csv_path, jsonl_path, max_items=None):
    df = pd.read_csv(csv_path)
    required = {'ID', 'clean_wav', 'noisy_wav'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f'{csv_path} thiếu cột: {sorted(missing)}')
    if max_items is not None:
        df = df.head(int(max_items))

    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    with jsonl_path.open('w', encoding='utf-8') as f:
        for row in df.to_dict('records'):
            item = {
                'id': row['ID'],
                'mixture': row['noisy_wav'],
                'target': row['clean_wav'],
                'noisy_wav': row['noisy_wav'],
                'clean_wav': row['clean_wav'],
            }
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    return jsonl_path, len(df)

metadata_base = Path(globals().get('data_dir', str(WORKSPACE / 'data/voicebank-demand')))
csv_manifests = {
    'train': metadata_base / 'train.csv',
    'valid': metadata_base / 'valid.csv',
    'test': metadata_base / 'test.csv',
}
jsonl_manifests = {}
if all(path.exists() for path in csv_manifests.values()):
    jsonl_manifests['train'], train_count = csv_to_jsonl(csv_manifests['train'], phase1_manifest_dir / 'train.jsonl')
    jsonl_manifests['valid'], valid_count = csv_to_jsonl(csv_manifests['valid'], phase1_manifest_dir / 'valid.jsonl')
    jsonl_manifests['test'], test_count = csv_to_jsonl(
        csv_manifests['test'],
        phase1_manifest_dir / 'test.jsonl',
        max_items=PHASE1_EVAL_MAX_TEST_ITEMS,
    )
    print('Manifest Phase 1:')
    print(f'  train: {jsonl_manifests["train"]} ({train_count} mẫu)')
    print(f'  valid: {jsonl_manifests["valid"]} ({valid_count} mẫu)')
    print(f'  test : {jsonl_manifests["test"]} ({test_count} mẫu)')
elif PHASE1_RUN_TRAINING or PHASE1_RUN_QUALITY_EVAL:
    raise FileNotFoundError('Thiếu train/valid/test CSV. Hãy chạy cell tạo metadata trước.')

def find_best_checkpoint(explicit_path=None):
    if explicit_path:
        path = Path(explicit_path)
        if not path.exists():
            raise FileNotFoundError(f'Không tìm thấy PHASE1_BASELINE_CHECKPOINT: {path}')
        return path

    roots = [WORKSPACE, Path('/content'), Path('/content/drive/MyDrive')]
    candidates = []
    for root in roots:
        if root.exists():
            candidates.extend(root.rglob('best.pt'))
        if candidates:
            break
    if not candidates:
        return None

    def score(path):
        lowered = ' '.join(part.lower() for part in path.parts)
        return (0 if 'deepvqe' in lowered else 1, len(str(path)), str(path))

    return sorted(candidates, key=score)[0]

phase1_baseline_ckpt = find_best_checkpoint(PHASE1_BASELINE_CHECKPOINT)
if phase1_baseline_ckpt is not None:
    print(f'Baseline checkpoint để eval so sánh: {phase1_baseline_ckpt}')
else:
    print('Không tìm thấy Baseline best.pt trong Drive/Colab; Phase 1 chỉ eval các checkpoint vừa train.')

def phase1_checkpoint_for_config(config_id):
    if config_id == 'Baseline' and phase1_baseline_ckpt is not None:
        return phase1_baseline_ckpt
    return phase1_runs_dir / config_id / 'best.pt'

if PHASE1_RUN_SMOKE_TESTS:
    run_py(['ablation/test_ablation_baseline.py'])
    run_py(['ablation/test_ablation_streaming.py', '--configs', *PHASE1_ARCH_CONFIGS])

if PHASE1_RUN_ARCH_BENCHMARK:
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', phase1_arch_csv,
        '--configs', *PHASE1_ARCH_CONFIGS,
        '--device', PHASE1_DEVICE,
        '--frames', PHASE1_ARCH_FRAMES,
        '--warmup', PHASE1_ARCH_WARMUP,
        '--repeats', PHASE1_ARCH_REPEATS,
    ])

if PHASE1_RUN_TRAINING:
    for config_id in PHASE1_TRAIN_CONFIGS:
        output_dir = phase1_runs_dir / config_id
        args = [
            'ablation/train_ablation.py',
            '--config-id', config_id,
            '--train-manifest', jsonl_manifests['train'],
            '--valid-manifest', jsonl_manifests['valid'],
            '--output-dir', output_dir,
            '--device', PHASE1_DEVICE,
            '--epochs', PHASE1_EPOCHS,
            '--batch-size', PHASE1_BATCH_SIZE,
            '--num-workers', PHASE1_NUM_WORKERS,
            '--early-stop-patience', PHASE1_EARLY_STOP_PATIENCE,
            '--early-stop-min-delta', PHASE1_EARLY_STOP_MIN_DELTA,
            '--early-stop-min-epochs', PHASE1_EARLY_STOP_MIN_EPOCHS,
        ]
        last_ckpt = output_dir / 'last.pt'
        if not PHASE1_PIN_MEMORY:
            args += ['--no-pin-memory']
        if PHASE1_USE_DATA_PARALLEL:
            args += ['--data-parallel']
        if PHASE1_RESUME_EXISTING and last_ckpt.exists():
            args += ['--resume', last_ckpt, '--ignore-bad-resume']
        run_py(args)

if PHASE1_RUN_QUALITY_EVAL:
    eval_configs = list(PHASE1_TRAIN_CONFIGS)
    if PHASE1_EVAL_BASELINE_IF_FOUND and phase1_baseline_ckpt is not None:
        eval_configs = ['Baseline', *eval_configs]

    for config_id in eval_configs:
        ckpt = phase1_checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua eval {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/eval_ablation_quality.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--manifest', jsonl_manifests['test'],
            '--output', phase1_quality_csv,
            '--device', PHASE1_DEVICE,
        ])

if PHASE1_RUN_ONNX_EXPORT:
    for config_id in PHASE1_TRAIN_CONFIGS:
        ckpt = phase1_checkpoint_for_config(config_id)
        if not ckpt.exists():
            print(f'Bỏ qua ONNX {config_id}: chưa có checkpoint {ckpt}')
            continue
        run_py([
            'ablation/export_ablation_onnx.py',
            '--config-id', config_id,
            '--checkpoint', ckpt,
            '--output-dir', phase1_onnx_dir,
            '--results', phase1_onnx_csv,
            '--device', 'cpu',
        ])

run_py([
    'ablation/collect_ablation_results.py',
    '--arch', phase1_arch_csv,
    '--quality', phase1_quality_csv,
    '--onnx', phase1_onnx_csv,
    '--output', phase1_summary_csv,
])

print('\nCSV Phase 1:')
for csv_path in [phase1_arch_csv, phase1_quality_csv, phase1_onnx_csv, phase1_summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path).head(30))

print('\nCheckpoint Phase 1:')
for config_id in PHASE1_TRAIN_CONFIGS:
    output_dir = phase1_runs_dir / config_id
    print(f'  {config_id}: best={output_dir / "best.pt"} | last={output_dir / "last.pt"}')

archive_base = Path('/content') / f'{PHASE1_RUN_TAG}_results'
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(phase1_results_dir))
print(f'Đã nén CSV Phase 1: {archive_path}')
display(FileLink(str(archive_path)))

## 11. Tải checkpoint Phase 1 về máy

Cell này nén `best.pt`, `last.pt` và `config.json` của các config Phase 1 để tải về máy.

In [ ]:
from pathlib import Path
import shutil

from IPython.display import FileLink, display

PHASE1_RUN_TAG = globals().get('PHASE1_RUN_TAG', 'phase1_b1b_vctk_demand_e80_bs8')
PHASE1_TRAIN_CONFIGS = globals().get('PHASE1_TRAIN_CONFIGS', ['B1b'])
workspace_dir = globals().get('workspace_dir', None)
if workspace_dir is None:
    workspace_dir = globals().get('WORKSPACE', Path('/content/drive/MyDrive/DeepVQE_Workspace'))
workspace_dir = Path(workspace_dir)

phase1_runs_dir = workspace_dir / 'runs' / 'ablation' / PHASE1_RUN_TAG
if not phase1_runs_dir.exists():
    raise FileNotFoundError(f'Không tìm thấy thư mục checkpoint Phase 1: {phase1_runs_dir}')

missing = []
for config_id in PHASE1_TRAIN_CONFIGS:
    for filename in ['best.pt', 'last.pt', 'config.json']:
        path = phase1_runs_dir / config_id / filename
        if not path.exists():
            missing.append(str(path))

if missing:
    print('Cảnh báo: thiếu một số file checkpoint/artifact:')
    for item in missing:
        print('  ' + item)

archive_base = Path('/content') / f'{PHASE1_RUN_TAG}_checkpoints'
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()

shutil.make_archive(str(archive_base), 'zip', root_dir=str(phase1_runs_dir))
print(f'Đã nén checkpoint Phase 1: {archive_path}')
print(f'Nguồn: {phase1_runs_dir}')
display(FileLink(str(archive_path)))

## 12. Đánh giá chất lượng mô hình: PESQ, STOI, SI-SDR, RTF

Cell này chạy đánh giá toàn diện trên **toàn bộ test set** với 4 tiêu chí:
- **PESQ** (Perceptual Evaluation of Speech Quality) — ITU-T P.862, wideband mode
- **STOI** (Short-Time Objective Intelligibility) — đo độ rõ ràng giọng nói
- **SI-SDR** (Scale-Invariant Signal-to-Distortion Ratio) — đo chất lượng tín hiệu (dB)
- **RTF** (Real-Time Factor) — thời gian xử lý / thời gian tín hiệu (< 1.0 = real-time)

**Lưu ý**: Cần chạy cell 5 (YAML config) và cell 6 (training) trước để có `hparams` và model checkpoint.

In [ ]:
# ============================================================
# 12. Đánh giá PESQ / STOI / SI-SDR / RTF trên test set
# ============================================================

# --- Cài đặt thư viện đánh giá nếu chưa có ---
import subprocess, sys
for pkg in ['pesq', 'pystoi']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import os
import time
import numpy as np
import pandas as pd
import torch
import torchaudio
from pesq import pesq as pesq_score
from pystoi import stoi as stoi_score
from tqdm.auto import tqdm
import IPython.display as ipd

# ===================== TÙY CHỈNH =====================
EVAL_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EVAL_SR = 16000          # Sample rate cho PESQ/STOI (phải = 16000)
EVAL_MAX_SAMPLES = None  # None = full test set; đặt số nhỏ (e.g. 20) để smoke test
EVAL_SAVE_CSV = True     # Lưu kết quả ra CSV trên Drive
# =====================================================

# --- Hàm tính SI-SDR ---
def compute_si_sdr(reference, estimation):
    """Tính Scale-Invariant Signal-to-Distortion Ratio (dB).
    reference, estimation: numpy arrays 1-D, cùng length.
    """
    ref = reference - np.mean(reference)
    est = estimation - np.mean(estimation)
    
    dot = np.sum(ref * est)
    s_ref_energy = np.sum(ref ** 2) + 1e-8
    
    s_target = (dot / s_ref_energy) * ref
    e_noise = est - s_target
    
    si_sdr = 10 * np.log10(np.sum(s_target ** 2) / (np.sum(e_noise ** 2) + 1e-8) + 1e-8)
    return float(si_sdr)

# --- Khôi phục hparams nếu kernel bị restart ---
if 'hparams' not in globals():
    print('Biến hparams chưa được định nghĩa. Đang tải lại từ hparams_vctk.yaml...')
    import sys, os
    from hyperpyyaml import load_hyperpyyaml
    work_dir = globals().get('WORK_DIR', '/content/drive/MyDrive/DeepVQE_Workspace')
    repo_dir = os.path.join(work_dir, 'deepvqe')
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
        if repo_dir not in sys.path:
            sys.path.insert(0, repo_dir)
    else:
        print('Cảnh báo: Không tìm thấy thư mục mã nguồn. Vui lòng chạy lại các Cell trên cùng để mount Drive và clone repo.')
    try:
        import deepvqe
        with open('hparams_vctk.yaml') as fin:
            hparams = load_hyperpyyaml(fin)
        if 'checkpointer' in hparams:
            print('Khôi phục trọng số mô hình từ checkpoint...')
            hparams['checkpointer'].recover_if_possible()
    except FileNotFoundError:
        print('Lỗi: Không tìm thấy file hparams_vctk.yaml! Vui lòng chạy lại Cell 5.')
        raise
    except ImportError as e:
        print('Lỗi import thư viện:', e)
        print('Vui lòng đảm bảo bạn đã chạy Cell 1 để cài đặt các thư viện (einops, speechbrain...)')
        raise
# --- Load model ---
print('Đang load model...')
model = hparams['model'].eval().to(EVAL_DEVICE)
n_fft = hparams.get('n_fft', 512)
hop_length = hparams.get('hop_length', 256)
win_length = hparams.get('win_length', 512)
window = torch.hann_window(win_length).to(EVAL_DEVICE)

# Đếm tham số
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model params: {total_params:,} (trainable: {trainable_params:,})')
print(f'Device: {EVAL_DEVICE}')

# --- Load test set ---
test_csv_path = hparams.get('test_annotation', 
    os.path.join(globals().get('data_dir', '/content/drive/MyDrive/DeepVQE_Workspace/data/voicebank-demand'), 'test.csv'))
df_test = pd.read_csv(test_csv_path)
if EVAL_MAX_SAMPLES is not None:
    df_test = df_test.head(EVAL_MAX_SAMPLES)
print(f'Test samples: {len(df_test)}')

# --- Inference + Metrics ---
results = []
total_audio_duration = 0.0
total_processing_time = 0.0

with torch.no_grad():
    for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc='Evaluating'):
        # Load audio
        noisy_wav, sr_n = torchaudio.load(row['noisy_wav'])
        clean_wav, sr_c = torchaudio.load(row['clean_wav'])
        
        # Resample nếu cần
        if sr_n != EVAL_SR:
            noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, EVAL_SR)
        if sr_c != EVAL_SR:
            clean_wav = torchaudio.functional.resample(clean_wav, sr_c, EVAL_SR)
        
        noisy_wav = noisy_wav.squeeze(0)  # [T]
        clean_wav = clean_wav.squeeze(0)  # [T]
        
        audio_duration = noisy_wav.shape[0] / EVAL_SR
        total_audio_duration += audio_duration
        
        # === Inference với đo RTF ===
        noisy_gpu = noisy_wav.to(EVAL_DEVICE)
        
        if EVAL_DEVICE == 'cuda':
            torch.cuda.synchronize()
        t_start = time.perf_counter()
        
        stft_in = torch.stft(
            noisy_gpu, n_fft=n_fft, hop_length=hop_length,
            win_length=win_length, window=window, return_complex=True
        )
        stft_real = torch.view_as_real(stft_in).unsqueeze(0)  # (1, F, T, 2)
        pred_stft = model(stft_real)
        pred_complex = torch.complex(pred_stft[..., 0], pred_stft[..., 1]).squeeze(0)
        enhanced = torch.istft(
            pred_complex, n_fft=n_fft, hop_length=hop_length,
            win_length=win_length, window=window, length=noisy_gpu.shape[0]
        )
        
        if EVAL_DEVICE == 'cuda':
            torch.cuda.synchronize()
        t_end = time.perf_counter()
        
        processing_time = t_end - t_start
        total_processing_time += processing_time
        rtf = processing_time / audio_duration
        
        # Chuyển về numpy
        enhanced_np = enhanced.cpu().numpy()
        clean_np = clean_wav.numpy()
        noisy_np = noisy_wav.numpy()
        
        # Align length
        min_len = min(len(enhanced_np), len(clean_np))
        enhanced_np = enhanced_np[:min_len]
        clean_np = clean_np[:min_len]
        noisy_np = noisy_np[:min_len]
        
        # === PESQ (wideband, 16kHz) ===
        try:
            pesq_enhanced = pesq_score(EVAL_SR, clean_np, enhanced_np, 'wb')
            pesq_noisy = pesq_score(EVAL_SR, clean_np, noisy_np, 'wb')
        except Exception as e:
            pesq_enhanced = float('nan')
            pesq_noisy = float('nan')
        
        # === STOI ===
        try:
            stoi_enhanced = stoi_score(clean_np, enhanced_np, EVAL_SR, extended=False)
            stoi_noisy = stoi_score(clean_np, noisy_np, EVAL_SR, extended=False)
        except Exception as e:
            stoi_enhanced = float('nan')
            stoi_noisy = float('nan')
        
        # === SI-SDR ===
        si_sdr_enhanced = compute_si_sdr(clean_np, enhanced_np)
        si_sdr_noisy = compute_si_sdr(clean_np, noisy_np)
        
        results.append({
            'ID': row['ID'],
            'PESQ_enhanced': round(pesq_enhanced, 4),
            'PESQ_noisy': round(pesq_noisy, 4),
            'PESQ_improvement': round(pesq_enhanced - pesq_noisy, 4) if not (np.isnan(pesq_enhanced) or np.isnan(pesq_noisy)) else float('nan'),
            'STOI_enhanced': round(stoi_enhanced, 4),
            'STOI_noisy': round(stoi_noisy, 4),
            'STOI_improvement': round(stoi_enhanced - stoi_noisy, 4) if not (np.isnan(stoi_enhanced) or np.isnan(stoi_noisy)) else float('nan'),
            'SI_SDR_enhanced_dB': round(si_sdr_enhanced, 2),
            'SI_SDR_noisy_dB': round(si_sdr_noisy, 2),
            'SI_SDR_improvement_dB': round(si_sdr_enhanced - si_sdr_noisy, 2),
            'RTF': round(rtf, 6),
            'duration_s': round(audio_duration, 3),
        })

# --- Tổng hợp kết quả ---
df_results = pd.DataFrame(results)

# Aggregate stats
overall_rtf = total_processing_time / total_audio_duration
summary = {
    'Metric': ['PESQ (enhanced)', 'PESQ (noisy)', 'PESQ (Δ improvement)',
               'STOI (enhanced)', 'STOI (noisy)', 'STOI (Δ improvement)',
               'SI-SDR enhanced (dB)', 'SI-SDR noisy (dB)', 'SI-SDR Δ (dB)',
               'RTF (mean)', 'RTF (overall)', 'Real-time capable?'],
    'Value': [
        f"{df_results['PESQ_enhanced'].mean():.4f}",
        f"{df_results['PESQ_noisy'].mean():.4f}",
        f"{df_results['PESQ_improvement'].mean():.4f}",
        f"{df_results['STOI_enhanced'].mean():.4f}",
        f"{df_results['STOI_noisy'].mean():.4f}",
        f"{df_results['STOI_improvement'].mean():.4f}",
        f"{df_results['SI_SDR_enhanced_dB'].mean():.2f}",
        f"{df_results['SI_SDR_noisy_dB'].mean():.2f}",
        f"{df_results['SI_SDR_improvement_dB'].mean():.2f}",
        f"{df_results['RTF'].mean():.6f}",
        f"{overall_rtf:.6f}",
        '✅ Có' if overall_rtf < 1.0 else '❌ Không',
    ]
}
df_summary = pd.DataFrame(summary)

print('\n' + '=' * 60)
print('  KẾT QUẢ ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH DeepVQE')
print('=' * 60)
print(f'Test samples: {len(df_results)}')
print(f'Model params: {total_params:,}')
print(f'Device: {EVAL_DEVICE}')
print(f'Total audio: {total_audio_duration:.1f}s | Processing: {total_processing_time:.1f}s')
print()
display(df_summary)

print('\n--- Chi tiết 10 mẫu đầu tiên ---')
display(df_results.head(10))

# --- Lưu CSV ---
if EVAL_SAVE_CSV:
    from pathlib import Path
    save_dir = Path('/content/drive/MyDrive/DeepVQE_Workspace/results/evaluation')
    save_dir.mkdir(parents=True, exist_ok=True)
    
    detail_csv = save_dir / 'eval_metrics_per_sample.csv'
    summary_csv_path = save_dir / 'eval_metrics_summary.csv'
    
    df_results.to_csv(detail_csv, index=False)
    df_summary.to_csv(summary_csv_path, index=False)
    
    print(f'\n📁 Đã lưu kết quả chi tiết:  {detail_csv}')
    print(f'📁 Đã lưu bảng tổng hợp:     {summary_csv_path}')
    
    from IPython.display import FileLink
    display(FileLink(str(detail_csv)))
    display(FileLink(str(summary_csv_path)))

In [ ]:
# Đo lường độ trễ streaming (RTF) trên từng frame
# Đảm bảo bạn đã mount Google Drive và lưu model vào thư mục Workspace tương ứng
import os
CHECKPOINT_PATH = '/content/drive/MyDrive/DeepVQE_Workspace/models/ablation_b1b/best_model.pt'
AUDIO_PATH = 'test_sample.wav' # Thay bằng file wav thực tế

!python ablation/test_streaming_latency.py --input {AUDIO_PATH} --config B1b --checkpoint {CHECKPOINT_PATH} --output streaming_metrics_b1b.csv

# Xem qua 5 dòng đầu tiên của file log kết quả
import pandas as pd
if os.path.exists('streaming_metrics_b1b.csv'):
    df = pd.read_csv('streaming_metrics_b1b.csv')
    display(df.head())